# Display checks

This notebook runs the display checks interactively. Each section creates a small ensemble, checks the returned text, and prints the result for review.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'src').is_dir():
    project_root = project_root.parent
if not (project_root / 'src' / 'ensemblelab').is_dir():
    raise RuntimeError('Open this notebook from within the ensemblelab repository.')
sys.path.insert(0, str(project_root / 'src'))

from ensemblelab import Ensemble, generate
from ensemblelab.optimizers.mmff import MMFFOptimizer


## Default ensemble display

An unoptimized ensemble reports the molecule summary and conformer table. Energy and convergence fields are unavailable until optimization.

In [2]:
ensemble = generate('CCO', n_confs=2)
result = ensemble.show()

assert 'Ensemble' in result
assert 'Energy: uncomputed' in result
assert 'Conformers' in result
assert 'N/A' in result

print(result)


Ensemble
----------------------------
SMILES: CCO
Atoms: 9
Conformers: 2
Energy: uncomputed
Optimization: not run

Conformers
------------------------------------
ID  Energy  Method  Converged
--  ------  ------  ---------
0   N/A     N/A     N/A
1   N/A     N/A     N/A


## History and metadata sections

History and raw metadata can be requested together. The conformer table can be omitted when the additional sections are the focus.

In [3]:
result = ensemble.show(history=True, metadata=True, conformers=False)

assert 'Workflow History' in result
assert '1. generation' in result
assert 'Metadata' in result
assert 'Conformers\n' not in result

print(result)


Ensemble
----------------------------
SMILES: CCO
Atoms: 9
Conformers: 2
Energy: uncomputed
Optimization: not run

Workflow History

1. generation
   Method: ETKDGv3
   Requested: 2
   Generated: 2
   Random seed: 42

Metadata
Key                  Value
-------------------  ----------------------------------------------------------------------------------------------------------------------------
n_conformers         2
optimization_status  unoptimized
energy_status        uncomputed
energy_unit          N/A
rdkit_version        2026.03.1
processing_history   process=generation, method=ETKDGv3, requested_smiles=CCO, canonical_smiles=CCO, n_requested=2, n_generated=2, random_seed=42


## Conformer display

A conformer display reports fields held directly by the conformer object. This case uses water with one generated conformer.

In [ ]:
conformer = generate('O', n_confs=1).conformers[0]
result = conformer.show()

assert 'Conformer 0' in result
assert 'Energy          N/A' in result
assert 'Atoms           3' in result

print(result)


## Optimized ensemble display

MMFF assigns energies, methods, and convergence states. The ensemble table reports relative energy from the lowest stored energy, and history records the generation and optimization events.

In [ ]:
optimized = MMFFOptimizer().optimize(generate('CCO', n_confs=2))
result = optimized.show(history=True)

assert 'Delta E (kcal/mol)' in result
assert 'Optimization: MMFF' in result
assert '1. generation' in result
assert '2. optimization' in result

print(result)


## Functional optimizer history

The display also reads optimization records stored under `optimization_history`. This is the provenance format used by the functional optimization API.

In [4]:
generated = generate('O', n_confs=1)
legacy_history_ensemble = Ensemble(
    smiles=generated.smiles,
    molecule=generated.molecule,
    conformers=generated.conformers,
    metadata={
        'optimization_history': [{'method': 'MMFF', 'max_steps': 500}],
    },
)
result = legacy_history_ensemble.show(history=True, conformers=False)

assert '1. optimization' in result
assert 'Method: MMFF' in result

print(result)


Ensemble
----------------------------
SMILES: O
Atoms: 3
Conformers: 1
Energy: uncomputed
Optimization: not run

Workflow History

1. optimization
   Method: MMFF
   Max steps: 500
